Accompanying Code for: <br>
**Johannes Zeitler, Meinard Müller. "A Unified Perspective on CTC and SDTW using Differentiable DTW", submitted to IEEE Transactions on Audio, Speech, and Language Processing, 2025.**

Johannes Zeitler (johannes.zeitler@audiolabs-erlangen.de), 2025

### Description
Load MTD annotations and compute/save strong targets, weak targets, and CTC targets.

In [31]:
import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
from hcqt import compute_hcqt, compute_hopsize_cqt
from tqdm.notebook import tqdm
import pandas as pd

from libfmp import b, c1

In [32]:
_, fs_cqt = compute_hopsize_cqt(25, 22050, 6)
print("operate on frame rate %.8fHz"%(fs_cqt))

operate on frame rate 24.60937500Hz


In [1]:
# the original CTC-MTD Paper (Zalkow and Mueller, 2021) defines particular data splits.
def get_files_for_split(path, idx):
    files = []
    for i in idx:
        split_in = pd.read_csv(os.path.join(path, "MTD_split5-%i.csv"%(i)))

        for _, row in split_in.iterrows():
            files.append(row[0].split(".")[0])
    return files

In [ ]:
MTD_dir = "path_to_MTD"
wav_dir = os.path.join(MTD_dir, "data_AUDIO")

# when we have all files
#all_wavs = [f.split(".")[0] for f in os.listdir(wav_dir) if ".wav" in f]
#all_wavs.sort()

# when we only use some files
all_wavs = []
for split in range(1,6,1):
    split_files = get_files_for_split("./data/MTD_splits", [split])
    all_wavs += [split_files[0]]
    all_wavs += [split_files[-1]]

metadata_dir = os.path.join(MTD_dir, "data_META")
CSV_dir = os.path.join(MTD_dir, "data_EDM-alig_CSV")
MIDI_dir = os.path.join(MTD_dir, "data_EDM-alig_MID")

In [56]:
# convert a list of notes to a 12xT chromagram
def list_to_chromagram(note_list, n_frames, fs):
    C = np.zeros((12, n_frames))

    for row in note_list:
        st = int(row[0]*fs)
        en = int((row[0]+row[1])*fs)
        pi = int(row[2]%12)

        C[pi, st:en] = 1
    return C

In [ ]:
for file in tqdm(all_wavs):
    hcqt_in = np.load(os.path.join("data", "hcqt", file+".npy"))
    n_frames = hcqt_in.shape[1]

    # load note list and transpose, if necessary
    metadata_in = pd.read_json(os.path.join(metadata_dir, file+".json"), typ='series').to_frame()
    transp = int(metadata_in[0].MidiTransposition)
    csv_in = pd.read_csv(os.path.join(CSV_dir, file+".csv"), sep=";")
    note_list = [[row.Start, row.Duration, row.Pitch + transp, 128, "Piano"] for _, row in csv_in.iterrows()]

    # ensure the theme is monophonic
    note_list_corr = []
    for i in range(len(note_list)-1):
        curr_start = note_list[i][0]
        curr_dur = note_list[i][1]    
        next_start = note_list[i+1][0]    
        max_dur = min([curr_dur, next_start-curr_start])    
        note_list_corr.append([curr_start, max_dur, note_list[i][2], note_list[i][3], note_list[i][4]])

    # strong targets (12xN)
    chromagram_strong = list_to_chromagram(note_list_corr, n_frames, fs_cqt)

    # weak targets (12xM)
    diffs = np.sum(np.abs(chromagram_strong[:,:-1] - chromagram_strong[:,1:]), axis=0) > 0
    diffs = np.concatenate([np.array([True]), diffs])
    chromagram_weak = chromagram_strong[:,diffs]

    # CTC targets [1,12]^M
    CTC_targets_chroma = np.array([int(row[2]%12) for row in note_list])

    # uniformly stretched weak targets
    stretch_indices = np.linspace(0,chromagram_weak.shape[1],chromagram_strong.shape[1], endpoint=False).astype(int)
    chromagram_weak_stretched = chromagram_weak[:,stretch_indices]

    np.save(os.path.join("data", "strong_targets_chroma", file+".npy"), arr=chromagram_strong)
    np.save(os.path.join("data", "weak_targets_chroma", file+".npy"), arr=chromagram_weak)
    np.save(os.path.join("data", "CTC_targets_chroma", file+".npy"), arr=CTC_targets_chroma)
    np.save(os.path.join("data", "weak_targets_stretched_chroma", file+".npy"), arr=chromagram_weak_stretched)